## Prepare a full knowledge graph dataset


We need to prepare the training, validation and external test datasets. We will use the training dataset to train the model and the test dataset to evaluate the model for all KGE models.

### [Require to Modify According to Your Situation] Prepare all relation files

We design a strategy to allow users to integrate their expected relation files for different purposes. Such as you might want to include the `malacards_mecfs` dataset when you want to train a model for ME/CFS.

In [3]:
import os

root_dir = os.path.dirname(os.getcwd())

# ---------- Parameters [Must be modified based on your situation] ----------
# dataset_name = "biomedgps-full-v20240127"
dataset_name = "biomedgps"
dataset_version = "v20250928"
skip_rows_not_in_entity_file = True
# The directory names must be consistent with the subdirectories in the formatted_relations folder.
blacklist_databases = []

# It's an optional parameter, if you don't want to split the dataset, you can ignore it.
#split_ratio = 0.8

# Which column will be kept in the final formatted file
relation_type_column = "formatted_relation_type"

# Which file will be used to format the relation types
relation_type_file = "relation_types.tsv"

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [4]:
root_dir

'/Users/zhuzhixing/KG/biomedgps-data'

In [5]:
# ---------- Load data ----------
graph_data_dir = os.path.join(root_dir, "graph_data")
formatted_relation_dir = os.path.join(graph_data_dir, "formatted_relations")

files = []
for dir in os.listdir(formatted_relation_dir):
    dir_path = os.path.join(formatted_relation_dir, dir)
    if os.path.isdir(dir_path):  # 检查是否为目录
        for file in os.listdir(dir_path):
            if file.endswith(".tsv") and file.startswith("formatted_") and dir not in blacklist_databases:
                files.append(os.path.join(dir_path, file))

print("Merging the following files:")
print("\n".join(files))

entity_file = os.path.join(graph_data_dir, "entities.tsv")
print("Number of entities: {}".format(len(open(entity_file).readlines())))

Merging the following files:
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/repoDB/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/ttd/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/Fibromine/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/dgidb/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/drkg/formatted_drkg.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/biosnap_disease/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/FibROAD/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/customdb/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/monarch/formatted_customdb.tsv
/Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/cbcg/formatted_customdb.t

### Dependencies

In [6]:
import os
import sys

lib_dir = os.path.join(os.path.dirname(os.getcwd()), "lib")

print("Adding {} to sys.path".format(lib_dir))
sys.path.append(lib_dir)

Adding /Users/zhuzhixing/KG/biomedgps-data/lib to sys.path


In [30]:
repo_commit_id

'df0cec658dcb855f35bda63e176e9b9df7f8db79'

### Metadata for recording the steps

To save the essential information of all relation files for reproducibility.

In [8]:
from metadata import DatasetMetadata, check_repo_clean

# For getting the correct commit id, we need to check if the repo is clean. If not, you should commit your changes first.
check_repo_clean(file_suffix = ".py", raise_error=False)
repo_commit_id = os.popen("git rev-parse HEAD").read().strip()
repo_path = os.popen("git config --get remote.origin.url").read().strip()
outputdir = os.path.join(root_dir, "datasets", f"{dataset_name}-{dataset_version}-{repo_commit_id[:6]}")
os.makedirs(outputdir, exist_ok=True)

dataset_metadata = DatasetMetadata(
    repo_commit_id=repo_commit_id,
    repo_path=repo_path,
    dataset_name=dataset_name,
    dataset_version=dataset_version,
    data_files=files,
    metadata=None,
)

dataset_metadata.to_json(os.path.join(outputdir, "metadata.json"))

### Merge all relation files into one file

In [10]:
import os
import subprocess
import pandas as pd
import tempfile

temp_dir = tempfile.mkdtemp()

args = ["python3", os.path.join(lib_dir, "data.py"), "merge-files"]

for f in files:
    args.extend(["--input", f])

kg_file = os.path.join(temp_dir, "knowledge_graph.tsv")
args.extend(["--output", kg_file])

print("Running: {}".format(" ".join(args)))
args_str = " ".join(args)
!{args_str}

if os.path.exists(kg_file):
    df = pd.read_csv(kg_file, sep="\t")
    source_ids = df[["source_id", "source_type"]].drop_duplicates()
    source_ids.columns = ["id", "label"]
    target_ids = df[["target_id", "target_type"]].drop_duplicates()
    target_ids.columns = ["id", "label"]
    ids = pd.concat([source_ids, target_ids]).drop_duplicates()
    print("Number of unique entity ids: {}".format(len(ids)))
    print("Number of deduplicated relations: {}".format(len(df.drop_duplicates())))

    entities = pd.read_csv(entity_file, sep="\t")

    dataset_metadata.add_step(
        note="Merge all relation files into one file",
        entity_file_before=None,
        entity_file_after=entity_file,
        relation_file_before=None,
        relation_file_after=kg_file,
    )

Running: python3 /Users/zhuzhixing/KG/biomedgps-data/lib/data.py merge-files --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/repoDB/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/ttd/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/Fibromine/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/dgidb/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/drkg/formatted_drkg.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/biosnap_disease/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/FibROAD/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/customdb/formatted_customdb.tsv --input /Users/zhuzhixing/KG/biomedgps-data/graph_data/formatted_relations/mon

/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/ipykernel_87811/3759005214.py:21: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(kg_file, sep="\t")


Number of unique entity ids: 200451
Number of deduplicated relations: 44793507


/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/ipykernel_87811/3759005214.py:30: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  entities = pd.read_csv(entity_file, sep="\t")


In [11]:
kg_file

'/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/knowledge_graph.tsv'

### Add curation mapping step

In [21]:
import pandas as pd

# 读取全部数据
df = pd.read_csv(kg_file, sep="\t", low_memory=False)
df

,raw_source_id,raw_target_id,raw_source_type,raw_target_type,relation_type,resource,pmids,key_sentence,source_id,source_type,target_id,target_type,formatted_relation_type
0,DrugBank:DB00107,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB00107,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
1,DrugBank:DB00917,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB00917,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
2,DrugBank:DB01160,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB01160,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
3,DrugBank:DB12789,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB12789,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
4,DrugBank:DB03808,UMLS:C0000880,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB03808,Compound,MONDO:0005629,Disease,BioMedGPS::Treatment::Compound:Disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44793502,OMIM:613485,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0019171,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793503,OMIM:613677,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013359,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793504,OMIM:613980,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013513,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793505,OMIM:614098,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013572,Disease,BioMedGPS::AssociatedWith::Pathway:Disease


In [22]:
# 读取映射表（from_id, to_id 两列）
mapping = pd.read_excel("id_redirects.xlsx") 

# 构建字典 {from_id: to_id}
id_map = dict(zip(mapping["from_id"], mapping["to_id"]))

In [23]:
# 替换前的副本
before = df[["source_id", "target_id"]].copy()

# 替换 source_id 和 target_id
df[["source_id", "target_id"]] = df[["source_id", "target_id"]].replace(id_map)

# 统计发生变化的行数
changed_rows = (before != df[["source_id", "target_id"]]).any(axis=1).sum()

print(f"共替换了 {changed_rows} 行")

共替换了 748 行


### [Optional] Filter out the relations that are not matched with our requirements

We can follow the results generated by the graph_analysis.ipynb to decide which relations should be kept.

In [28]:
kg_file_ignore_relation_types_filtered = os.path.join(
    temp_dir, "knowledge_graph_ignore_relation_types_filtered.tsv"
)

In [24]:
print("Number of relations: {}".format(len(df)))
ignore_relation_types = [
    # Virus gene relations are not useful for our use case.
    "bioarx::Coronavirus_ass_host_gene::Disease:Gene",
    "bioarx::Covid2_acc_host_gene::Disease:Gene",
    "bioarx::DrugHumGen::Compound:Gene",
    "bioarx::DrugVirGen::Compound:Gene",
    "bioarx::HumGenHumGen::Gene:Gene",
    "bioarx::VirGenHumGen::Gene:Gene",
    # We don't like associated_with relation type.
    "PrimeKG::associated_with::Disease:Gene",
    "PrimeKG::associated_with::Gene:Disease",
    "PrimeKG::associated_with::Gene:Symptom",
    "PrimeKG::associated_with::Symptom:Gene",
    # We don't like ontology tree
    "PrimeKG::parent-child::Anatomy:Anatomy",
    "PrimeKG::parent-child::BiologicalProcess:BiologicalProcess",
    "PrimeKG::parent-child::CellularComponent:CellularComponent",
    "PrimeKG::parent-child::Disease:Disease",
    "PrimeKG::parent-child::MolecularFunction:MolecularFunction",
    "PrimeKG::parent-child::Pathway:Pathway",
    "PrimeKG::parent-child::Symptom:Symptom",
]

Number of relations: 44793507


In [25]:
ignore_relation_types

['bioarx::Coronavirus_ass_host_gene::Disease:Gene',
 'bioarx::Covid2_acc_host_gene::Disease:Gene',
 'bioarx::DrugHumGen::Compound:Gene',
 'bioarx::DrugVirGen::Compound:Gene',
 'bioarx::HumGenHumGen::Gene:Gene',
 'bioarx::VirGenHumGen::Gene:Gene',
 'PrimeKG::associated_with::Disease:Gene',
 'PrimeKG::associated_with::Gene:Disease',
 'PrimeKG::associated_with::Gene:Symptom',
 'PrimeKG::associated_with::Symptom:Gene',
 'PrimeKG::parent-child::Anatomy:Anatomy',
 'PrimeKG::parent-child::BiologicalProcess:BiologicalProcess',
 'PrimeKG::parent-child::CellularComponent:CellularComponent',
 'PrimeKG::parent-child::Disease:Disease',
 'PrimeKG::parent-child::MolecularFunction:MolecularFunction',
 'PrimeKG::parent-child::Pathway:Pathway',
 'PrimeKG::parent-child::Symptom:Symptom']

In [26]:
df = df[~df["relation_type"].isin(ignore_relation_types)]
print("Number of relations after removed ignore relation_types: {}".format(len(df)))

Number of relations after removed ignore relation_types: 44793419


In [31]:
relation_type_map = pd.read_csv(
    os.path.join(graph_data_dir, "relation_types.tsv"), sep="\t"
)
relation_type_map


,relation_type,raw_description,formatted_relation_type,description,prompt_template,resource,abbr,source_type,target_type,annotation
0,PrimeKG::parent-child::Anatomy:Anatomy,Parent-child relationship between anatomies,BioMedGPS::ParentChild::Anatomy:Anatomy,A hierarchical relationship between two anatom...,Is there a hierarchical relationship between t...,PrimeKG,parent-child,Anatomy,Anatomy,NaN
1,Hetionet::AdG::Anatomy:Gene,Anatomy downregulates the gene,BioMedGPS::E-::Anatomy:Gene,The gene is not significantly expressed in a s...,Is the #Gene# not significantly expressed in t...,Hetionet,AdG,Anatomy,Gene,Anatomy–downregulates–Gene
2,Hetionet::AeG::Anatomy:Gene,Anatomy expresses the gene,BioMedGPS::E::Anatomy:Gene,The gene has a general level of expression in ...,Does the #Gene# exhibit a general level of exp...,Hetionet,AeG,Anatomy,Gene,Anatomy–expresses–Gene
3,Hetionet::AuG::Anatomy:Gene,Anatomy upregulates the gene,BioMedGPS::E+::Anatomy:Gene,The gene is significantly expressed in a speci...,Is the #Gene# significantly expressed in the #...,Hetionet,AuG,Anatomy,Gene,Anatomy–upregulates–Gene
4,PrimeKG::expression_absent::Anatomy:Gene,Anatomy lacks expression of a gene,BioMedGPS::NE::Anatomy:Gene,The gene is not expressed in a specific anatom...,"Is the #Gene# not expressed in the #Anatomy#, ...",PrimeKG,expression_absent,Anatomy,Gene,NaN
...,...,...,...,...,...,...,...,...,...,...
397,biolink::subclass_of::Phenotype:Disease,A phenotype is considered a specific manifesta...,BioMedGPS::Present::Disease:Phenotype,A symptom that is present and characteristic o...,Is the #Symptom# characteristic and present in...,biolink,subclass_of,Phenotype,Disease,NaN
398,biolink::part_of::Protein:CellularComponent,The protein is structurally part of a larger c...,BioMedGPS::InCC::Gene:CellularComponent,A gene associated with a specific cellular com...,Is the #Gene# associated with the #CellularCom...,biolink,part_of,Protein,CellularComponent,NaN
399,biolink::caused_by::Disease:CellularComponent,The disease arises due to changes in a cellula...,BioMedGPS::Causer::CellularComponent:Disease,A change or dysfunction in a cellular componen...,Does the #CellularComponent# contribute to the...,biolink,caused_by,Disease,CellularComponent,NaN
400,biolink::subclass_of::CellularComponent:Anatomy,A cellular component is viewed as a part of br...,BioMedGPS::ParentChild::CellularComponent:Anatomy,A cellular component is a part of or derived f...,Is there a hierarchical relationship between #...,biolink,subclass_of,CellularComponent,Anatomy,NaN


In [32]:
relation_types = relation_type_map["relation_type"].tolist()
relation_types

['PrimeKG::parent-child::Anatomy:Anatomy',
 'Hetionet::AdG::Anatomy:Gene',
 'Hetionet::AeG::Anatomy:Gene',
 'Hetionet::AuG::Anatomy:Gene',
 'PrimeKG::expression_absent::Anatomy:Gene',
 'PrimeKG::expression_present::Anatomy:Gene',
 'PrimeKG::parent-child::BiologicalProcess:BiologicalProcess',
 'PrimeKG::interacts_with::BiologicalProcess:Gene',
 'PrimeKG::parent-child::CellularComponent:CellularComponent',
 'PrimeKG::interacts_with::CellularComponent:Gene',
 'DRUGBANK::x-atc::Compound:Atc',
 'DRUGBANK::ddi-interactor-in::Compound:Compound',
 'Hetionet::CrC::Compound:Compound',
 'PrimeKG::synergistic_interaction::Compound:Compound',
 'DRUGBANK::treats::Compound:Disease',
 'GNBR::C::Compound:Disease',
 'GNBR::J::Compound:Disease',
 'GNBR::Mp::Compound:Disease',
 'GNBR::Pa::Compound:Disease',
 'GNBR::Pr::Compound:Disease',
 'GNBR::Sa::Compound:Disease',
 'GNBR::T::Compound:Disease',
 'Hetionet::CpD::Compound:Disease',
 'Hetionet::CtD::Compound:Disease',
 'PrimeKG::contraindication::Compound

In [33]:
before_count = len(df)
before_count

44793419

In [34]:
before_count = len(df)

# Filter to only keep rows that are in the relation_types list
df = df[df["relation_type"].isin(relation_types)]

after_count = len(df)
removed_count = before_count - after_count

print(f"Number of relations before filtering: {before_count}")
print(f"Number of relations after filtering: {after_count}")
print(f"Number of relations removed: {removed_count}")


Number of relations before filtering: 44793419
Number of relations after filtering: 44701374
Number of relations removed: 92045


In [35]:
df

,raw_source_id,raw_target_id,raw_source_type,raw_target_type,relation_type,resource,pmids,key_sentence,source_id,source_type,target_id,target_type,formatted_relation_type
0,DrugBank:DB00107,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB00107,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
1,DrugBank:DB00917,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB00917,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
2,DrugBank:DB01160,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB01160,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
3,DrugBank:DB12789,UMLS:C0000810,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB12789,Compound,MESH:D000027,Disease,BioMedGPS::Treatment::Compound:Disease
4,DrugBank:DB03808,UMLS:C0000880,Compound,Disease,RepoDB::therapeutic::Compound:Disease,RepoDB,NaN,NaN,DrugBank:DB03808,Compound,MONDO:0005629,Disease,BioMedGPS::Treatment::Compound:Disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44793502,OMIM:613485,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0019171,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793503,OMIM:613677,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013359,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793504,OMIM:613980,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013513,Disease,BioMedGPS::AssociatedWith::Pathway:Disease
44793505,OMIM:614098,REACT:R-HSA-997272,Disease,Pathway,CTD::IS_ASSOCIATED_WITH::Disease:Pathway,CTD,NaN,NaN,REACT:R-HSA-997272,Pathway,MONDO:0013572,Disease,BioMedGPS::AssociatedWith::Pathway:Disease


In [36]:
ignore_formatted_relation_types = [
    # There are too much relations in this relation type, but they might not useful.
    "BioMedGPS::Interaction::Compound:Compound",
    # We don't like associated_with relation type.
    "BioMedGPS::AssociatedWith::Gene:Gene",
]

In [37]:
df = df[~df["formatted_relation_type"].isin(ignore_formatted_relation_types)]
print("Number of relations after removed ignore formatted_relation_types: {}".format(len(df)))

Number of relations after removed ignore formatted_relation_types: 42796799


In [38]:
df.to_csv(kg_file_ignore_relation_types_filtered, sep="\t", index=False)
kg_file = kg_file_ignore_relation_types_filtered
kg_file

'/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/knowledge_graph_ignore_relation_types_filtered.tsv'

### [Optional] Map all mouse genes to human genes as much as possible

#### Establish a mapping table between mice and humans

In [41]:
## Number of Mouse / Rat / Human Genes
entities = pd.read_csv(entity_file, sep="\t")
genes = entities[entities["label"] == "Gene"]
mouse_genes = genes[genes["taxid"] == 10090]
rat_genes = genes[genes["taxid"] == 10116]
human_genes = genes[genes["taxid"] == 9606]
print("Number of Entities: ", len(mouse_genes), len(rat_genes), len(human_genes))

/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/ipykernel_87811/3387816344.py:2: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  entities = pd.read_csv(entity_file, sep="\t")


In [45]:
knowledge_graph = pd.read_csv(kg_file, sep="\t")

mouse_relations = knowledge_graph[
    knowledge_graph["source_id"].isin(mouse_genes["id"])
    | knowledge_graph["target_id"].isin(mouse_genes["id"])
]

human_relations = knowledge_graph[
    knowledge_graph["source_id"].isin(human_genes["id"])
    | knowledge_graph["target_id"].isin(human_genes["id"])
]
print(f"Number of mouse gene relations: {len(mouse_relations)}, Number of human gene relations: {len(human_relations)}")

/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/ipykernel_87811/3875376327.py:1: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  knowledge_graph = pd.read_csv(kg_file, sep="\t")


Number of mouse gene relations: 2517961, Number of human gene relations: 31971689


In [46]:
human_mouse_gene_mappings = pd.read_csv(
    os.path.join(graph_data_dir, "mapping", "human_mouse_gene_mappings.tsv"), sep="\t"
)
# NOTE: There might be multiple mappings for a single mouse gene, we will use the first mapping for now. such as PTCD1[ENTREZ:26024] and ATP5MF-PTCD1[ENTREZ:100526740] have the same mouse gene mapping. Ptcd1[ENTREZ: 71799]. 
# human_mouse_gene_map["ENTREZ:71799"]
human_mouse_gene_map = dict(
    zip(
        human_mouse_gene_mappings["entrez_id_mouse"],
        human_mouse_gene_mappings["entrez_id_human"],
    )
)

In [47]:
human_mouse_gene_map

{'ENTREZ:23825': 'ENTREZ:8815',
 'ENTREZ:18585': 'ENTREZ:5152',
 'ENTREZ:66514': 'ENTREZ:80150',
 'ENTREZ:20480': 'ENTREZ:81570',
 'ENTREZ:13726': 'ENTREZ:2010',
 'ENTREZ:16952': 'ENTREZ:301',
 'ENTREZ:66477': 'ENTREZ:84833',
 'ENTREZ:338355': 'ENTREZ:23307',
 'ENTREZ:241116': 'ENTREZ:255101',
 'ENTREZ:665270': 'ENTREZ:151056',
 'ENTREZ:11766': 'ENTREZ:8906',
 'ENTREZ:17850': 'ENTREZ:4594',
 'ENTREZ:19386': 'ENTREZ:727851',
 'ENTREZ:620592': 'ENTREZ:27112',
 'ENTREZ:69870': 'ENTREZ:84265',
 'ENTREZ:14419': 'ENTREZ:51083',
 'ENTREZ:208043': 'ENTREZ:23067',
 'ENTREZ:21375': 'ENTREZ:10716',
 'ENTREZ:21385': 'ENTREZ:6909',
 'ENTREZ:27993': 'ENTREZ:92856',
 'ENTREZ:380718': 'ENTREZ:54903',
 'ENTREZ:56417': 'ENTREZ:103',
 'ENTREZ:217138': 'ENTREZ:79170',
 'ENTREZ:67979': 'ENTREZ:84896',
 'ENTREZ:56637': 'ENTREZ:2932',
 'ENTREZ:497652': 'ENTREZ:65057',
 'ENTREZ:71361': 'ENTREZ:84883',
 'ENTREZ:208211': 'ENTREZ:644974',
 'ENTREZ:52014': 'ENTREZ:116150',
 'ENTREZ:19892': 'ENTREZ:6121',
 'ENTREZ

#### Convert mouse genes to human genes

In [49]:
# We don't like mouse genes, let's convert them to human genes. If a mouse gene doesn't have a human gene mapping, we will keep the mouse gene. So the users can see that the gene is a mouse gene.
# Convert the mouse_genes["id"] Series to a set for faster lookup
mouse_gene_ids = set(mouse_genes["id"].values)

# Vectorized operation for source_id
knowledge_graph["source_id"] = knowledge_graph["source_id"].map(
    lambda x: human_mouse_gene_map.get(x, x) if x in mouse_gene_ids else x
)

# Vectorized operation for target_id
knowledge_graph["target_id"] = knowledge_graph["target_id"].map(
    lambda x: human_mouse_gene_map.get(x, x) if x in mouse_gene_ids else x
)

# Check whether the conversion is successful
# Extract all relations (`converted_mouse_relations`) that still contain a mouse gene ID after conversion. 
converted_mouse_relations = knowledge_graph[
    knowledge_graph["source_id"].isin(mouse_genes["id"])
    | knowledge_graph["target_id"].isin(mouse_genes["id"])
]
#Extract all relations (`converted_human_relations`) where either side is a human gene ID.
converted_human_relations = knowledge_graph[
    knowledge_graph["source_id"].isin(human_genes["id"])
    | knowledge_graph["target_id"].isin(human_genes["id"])
]

# We cannot use the pattern below because some gene names don't follow the pattern. for example, "Bdnf" is used as a human gene in GNBR database.
# pattern = r"^[A-Z][a-z]+$"
# not_matched_genes = knowledge_graph[
#     ((knowledge_graph["source_type"] == "Gene") & knowledge_graph["source_name"].str.match(pattern, na=False)) |
#     ((knowledge_graph["target_type"] == "Gene") & knowledge_graph["target_name"].str.match(pattern, na=False))
# ]
# not_matched_genes[
#     (not_matched_genes["source_id"] == "ENTREZ:627")
#     | (not_matched_genes["target_id"] == "ENTREZ:627")
# ]
#The variable `converted_not_matched_genes` contains all relations that do not include mouse IDs.
converted_not_matched_genes = knowledge_graph[
    ~(
        knowledge_graph["source_id"].isin(mouse_genes["id"])
        | knowledge_graph["target_id"].isin(mouse_genes["id"])
    )
]

# Expected: 0, xxx
print(
    "Mouse: ",
    len(mouse_relations),
    " Before /",
    len(converted_mouse_relations),
    " After /",
    len(converted_not_matched_genes),
    " Not matched",
)
print(
    "Human: ",
    len(human_relations),
    " Before /",
    len(converted_human_relations),
    " After",
)


Mouse:  2517961  Before / 656566  After / 42140233  Not matched
Human:  31971689  Before / 33788476  After


In [50]:
# Write the knowledge graph to a file
kg_file_mouse_converted = os.path.join(temp_dir, "knowledge_graph_mouse_converted.tsv")
knowledge_graph.to_csv(kg_file_mouse_converted, sep="\t", index=False)

dataset_metadata.add_step(
    note="Map all mouse genes to human genes as much as possible",
    entity_file_before=entity_file,
    entity_file_after=entity_file,
    relation_file_before=kg_file,
    relation_file_after=kg_file_mouse_converted,
)

kg_file = kg_file_mouse_converted

In [75]:
kg_file

'/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpyrhh8j0k/knowledge_graph_mouse_converted.tsv'

### [Optional] Format all relation types

#### Check whether there are any omissions in the mapping

In [51]:
if relation_type_file and os.path.exists(relation_type_file):
    relation_types = pd.read_csv(relation_type_file, sep="\t")

    print("Number of relation types: {}".format(len(relation_types)))

Number of relation types: 402


In [52]:
 # Read the kg file
knowledge_graph = pd.read_csv(kg_file, sep="\t")

/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/ipykernel_87811/3092633169.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  knowledge_graph = pd.read_csv(kg_file, sep="\t")


In [53]:
knowledge_graph_origin = knowledge_graph

In [62]:
no_biogpsmed_relations = knowledge_graph_origin[
    ~(
        knowledge_graph_origin["formatted_relation_type"].str.startswith("BioMedGPS:", na=False)
    )
]

biogpsmed_relations = knowledge_graph_origin[
        knowledge_graph_origin["formatted_relation_type"].str.startswith("BioMedGPS:", na=False)
]
# Print the number of entries and the type of relationship
print("Number of relations with relation_type not starting with 'BioMedGPS:':", len(no_biogpsmed_relations))
print(no_biogpsmed_relations["formatted_relation_type"].unique())

print("Number of relations with relation_type not starting with 'BioMedGPS:':", len(biogpsmed_relations))
print(biogpsmed_relations["formatted_relation_type"].unique()) 

Number of relations with relation_type not starting with 'BioMedGPS:': 14027
['DRUGBANK::treats::Compound:Disease' 'Hetionet::GpPW::Gene:Pathway']
Number of relations with relation_type not starting with 'BioMedGPS:': 42782772
['BioMedGPS::Treatment::Compound:Disease' 'BioMedGPS::E-::Disease:Gene'
 'BioMedGPS::E+::Disease:Gene' 'BioMedGPS::Agonist::Compound:Gene'
 'BioMedGPS::Modulator::Compound:Gene'
 'BioMedGPS::Inhibitor::Compound:Gene'
 'BioMedGPS::AllostericModulator::Compound:Gene'
 'BioMedGPS::Blocker::Compound:Gene' 'BioMedGPS::Binder::Compound:Gene'
 'BioMedGPS::Antibody::Compound:Gene'
 'BioMedGPS::Activator::Compound:Gene' 'BioMedGPS::E+::Compound:Gene'
 'BioMedGPS::AssociatedWith::Compound:Gene'
 'BioMedGPS::Antagonist::Compound:Gene'
 'BioMedGPS::SimilarWith::Disease:Disease'
 'BioMedGPS::SimilarWith::Compound:Compound'
 'BioMedGPS::InPC::Compound:PharmacologicClass'
 'BioMedGPS::Interaction::Gene:Gene' 'BioMedGPS::Influencer::Gene:Gene'
 'BioMedGPS::Binder::Gene:Gene' 'Bi

In [61]:
 # Format the relation types
    ## Remove the formatted_relation_type column if it exists
if "formatted_relation_type" in no_biogpsmed_relations.columns:
    no_biogpsmed_relations = no_biogpsmed_relations.drop(columns=["formatted_relation_type"])
    no_biogpsmed_relations = no_biogpsmed_relations.merge(relation_types[["relation_type", "formatted_relation_type"]], on="relation_type", how="left")
no_biogpsmed_relations

,raw_source_id,raw_target_id,raw_source_type,raw_target_type,relation_type,resource,pmids,key_sentence,source_id,source_type,target_id,target_type,formatted_relation_type
0,SYMBOL:CCR2,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:729230,Gene,MONDO:0957421,Disease,BioMedGPS::Treatment::Compound:Disease
1,SYMBOL:F9,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:2158,Gene,MONDO:0957421,Disease,BioMedGPS::Treatment::Compound:Disease
2,SYMBOL:HPD,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:3242,Gene,MONDO:0957421,Disease,BioMedGPS::Treatment::Compound:Disease
3,SYMBOL:KRT6A,ICD-11:1C80,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:3853,Gene,MONDO:0957421,Disease,BioMedGPS::Treatment::Compound:Disease
4,SYMBOL:APOH,ICD-11:4A45,Gene,Disease,DRUGBANK::treats::Compound:Disease,TTD,NaN,NaN,ENTREZ:350,Gene,MONDO:0007140,Disease,BioMedGPS::Treatment::Compound:Disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14022,SYMBOL:PTGFR,WikiPathways:WP98,Gene,Pathway,Hetionet::GpPW::Gene:Pathway,TTD,NaN,NaN,ENTREZ:5737,Gene,WikiPathways:WP98,Pathway,BioMedGPS::InPathway::Gene:Pathway
14023,SYMBOL:PTGIR,WikiPathways:WP98,Gene,Pathway,Hetionet::GpPW::Gene:Pathway,TTD,NaN,NaN,ENTREZ:5739,Gene,WikiPathways:WP98,Pathway,BioMedGPS::InPathway::Gene:Pathway
14024,SYMBOL:PTGS1,WikiPathways:WP98,Gene,Pathway,Hetionet::GpPW::Gene:Pathway,TTD,NaN,NaN,ENTREZ:5742,Gene,WikiPathways:WP98,Pathway,BioMedGPS::InPathway::Gene:Pathway
14025,SYMBOL:PTGS2,WikiPathways:WP98,Gene,Pathway,Hetionet::GpPW::Gene:Pathway,TTD,NaN,NaN,ENTREZ:5743,Gene,WikiPathways:WP98,Pathway,BioMedGPS::InPathway::Gene:Pathway


In [63]:
knowledge_graph_formatted_relation_type = pd.concat([biogpsmed_relations, no_biogpsmed_relations], ignore_index=True)
print("Number of combind knowledge graph: {}".format(len(knowledge_graph_formatted_relation_type)))

Number of combind knowledge graph: 42796799


#### Check whether there are any omissions in the mapping

In [65]:
###Check whether there are any omissions in the mapping
invalid_knowledge_graph = knowledge_graph_formatted_relation_type[knowledge_graph_formatted_relation_type["formatted_relation_type"].isna()]
print("Number of invalid knowledge graph: {}".format(len(invalid_knowledge_graph)))

invalid_knowledge_graph_file = os.path.join(temp_dir, "invalid_knowledge_graph.tsv")
invalid_knowledge_graph.to_csv(invalid_knowledge_graph_file, sep="\t", index=False)
print("Please check the invalid knowledge graph file: {}".format(invalid_knowledge_graph_file))


Number of invalid knowledge graph: 0
Please check the invalid knowledge graph file: /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/invalid_knowledge_graph.tsv


In [66]:
kg_file_relation_types_formatted = os.path.join(temp_dir, "knowledge_graph_relation_types_formatted.tsv")
knowledge_graph_formatted_relation_type.to_csv(kg_file_relation_types_formatted, sep="\t", index=False)

dataset_metadata.add_step(
    note="Format the relation types",
    entity_file_before=entity_file,
    entity_file_after=entity_file,
    relation_file_before=kg_file,
    relation_file_after=kg_file_relation_types_formatted,
)
    
kg_file = kg_file_relation_types_formatted

In [67]:
kg_file

'/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/knowledge_graph_relation_types_formatted.tsv'

### Annotate the knowledge graph with the entities

In [68]:
args = [
    "python3",
    os.path.join(os.path.dirname(lib_dir), "graph_data", "scripts", "annotate_relations.py"),
    "--entity-file",
    entity_file,
    "--relation-file",
    kg_file,
    "--output-dir",
    os.path.dirname(kg_file),
    "--strict-mode" if skip_rows_not_in_entity_file else "",
]

print("Running: {}".format(" ".join(args)))
args_str = " ".join(args)
!{args_str}
annotated_kg_file = os.path.join(temp_dir, "annotated_knowledge_graph.tsv")
print("File written to: {}".format(annotated_kg_file))

dataset_metadata.add_step(
    note="Annotate the knowledge graph with the entities",
    entity_file_before=entity_file,
    entity_file_after=entity_file,
    relation_file_before=kg_file,
    relation_file_after=annotated_kg_file,
)

Running: python3 /Users/zhuzhixing/KG/biomedgps-data/graph_data/scripts/annotate_relations.py --entity-file /Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv --relation-file /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/knowledge_graph_relation_types_formatted.tsv --output-dir /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt --strict-mode
Found 42796799 relations in the input file
You're in strict mode, so 0 relations were skipped.
File written to: /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/annotated_knowledge_graph.tsv


### Extract valid entities from the knowledge graph

In [69]:
import pandas as pd

knowledge_graph = pd.read_csv(kg_file, sep="\t", low_memory=False)
entities = pd.read_csv(entity_file, sep="\t", low_memory=False)

source_entities = knowledge_graph[["source_id", "source_type"]].drop_duplicates()
source_entities.columns = ["id", "label"]
target_entities = knowledge_graph[["target_id", "target_type"]].drop_duplicates()
target_entities.columns = ["id", "label"]
source_target_entities = pd.concat([source_entities, target_entities]).drop_duplicates()

valid_entities = pd.merge(source_target_entities, entities, on=["id", "label"], how="left", indicator=True)
knowledge_graph_entities_file = os.path.join(temp_dir, "knowledge_graph_entities.tsv")
valid_entities.to_csv(knowledge_graph_entities_file, sep="\t", index=False)

### Copy all files to the dataset folder

In [70]:
os.makedirs(outputdir, exist_ok=True)

files = [
    (entity_file, os.path.join(outputdir, "annotated_entities.tsv")),
    (kg_file, os.path.join(outputdir, "knowledge_graph.tsv")),
    (annotated_kg_file, os.path.join(outputdir, "annotated_knowledge_graph.tsv")),
    (knowledge_graph_entities_file, os.path.join(outputdir, "knowledge_graph_entities.tsv")),
]

In [71]:
for f, output_file in files:
    print("Copying {} to {}".format(f, output_file))
    subprocess.check_output(["cp", f, output_file])
    output_zip_file = output_file + ".zip"
    print("Zipping {} to {}".format(output_file, output_zip_file))
    subprocess.check_output(["zip", "-j", output_zip_file, output_file])
    print("Removing {}".format(output_file))
    subprocess.check_output(["rm", output_file])

print("Please found all files in {}".format(outputdir))

Copying /Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv to /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/annotated_entities.tsv
Zipping /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/annotated_entities.tsv to /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/annotated_entities.tsv.zip
Removing /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/annotated_entities.tsv
Copying /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmpj6_2hxgt/knowledge_graph_relation_types_formatted.tsv to /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph.tsv
Zipping /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph.tsv to /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph.tsv.zip
Removing /Users/zhuzhixing/KG/biomedgps-data/datasets/biomedgps-v20250928-df0cec/knowledge_graph.tsv
Copying /var/fol

### Upload the dataset to Dropbox and Dataverse

You can upload the dataset to Dropbox first, then select the files from Dropbox to upload to Dataverse.

https://dataverse.harvard.edu/dataverse/biomedgps